# Test_Bilstm

## imports

In [23]:
import pickle
import numpy as np
import re
import html
import tensorflow as tf
from keras.preprocessing.sequence import pad_sequences
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from ftfy import fix_text
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
import time
from datetime import datetime
import os
import praw
import pandas as pd
from tqdm import tqdm

# Download NLTK data if not already available
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Text Preprocessing: Negation Handling and Trigram Generation


In [ ]:
NEGATIONS = {
    "no", "nor", "not", "n't", "never", "none", "nobody", "nothing", 
    "nowhere", "hardly", "scarcely", "barely", "wouldn't", "couldn't", 
    "shouldn't", "won't", "can't", "don't", "doesn't", "didn't", 
    "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", 
    "hadn't", "without"
}

CUSTOM_STOPWORDS = set(stopwords.words('english'))
STOP_WORDS = CUSTOM_STOPWORDS - NEGATIONS

print(f"Stop words defined: {len(STOP_WORDS)} words (excluding {len(NEGATIONS)} negations)")

# %%
class TrigramTextPreprocessor(BaseEstimator, TransformerMixin):
    """
    Custom text preprocessor that:
    1. Cleans text (removes URLs, mentions, hashtags, etc.)
    2. Generates trigrams
    3. Converts to sequences and pads them
    """
    
    def __init__(self, tokenizer=None, max_len=100, stop_words=None):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stop_words = stop_words if stop_words is not None else STOP_WORDS
        self.stemmer = PorterStemmer()
        # Store vocabulary size for safety checks
        if tokenizer is not None:
            self.vocab_size = len(tokenizer.word_index) + 1  # +1 for OOV token

    def fit(self, X, y=None):
        return self

    def clean_text(self, text):
        """Applies all text cleaning steps"""
        if not isinstance(text, str):
            return ""

        text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
        text = re.sub(r'@\w+', '', text)  # Remove mentions
        text = re.sub(r'#\w+', '', text)  # Remove hashtags
        text = re.sub(r'\s+', ' ', text).strip()  # Normalize spaces
        text = text.lower()  # Convert to lowercase
        text = re.sub(r"[^a-z0-9\s.,!?']", '', text)  # Keep basic characters
        text = fix_text(text)  # Fix text encoding issues
        text = html.unescape(text)  # Unescape HTML entities
        text = re.sub(r'\b\d+\b', '', text)  # Remove isolated numbers
        text = re.sub(r'([!?.,])\1+', r'\1', text)  # Normalize punctuation
        
        # Remove stop words
        tokens = text.split()
        filtered_text = ' '.join([word for word in tokens if word not in self.stop_words])

        return filtered_text

    def generate_trigrams(self, text):
        """Generate trigrams from cleaned text"""
        tokens = word_tokenize(text)
        stemmed = [self.stemmer.stem(token) for token in tokens if token not in self.stop_words]
        trigram_tuples = list(ngrams(stemmed, 3))  # Generate actual trigrams
        trigram_strings = [' '.join(triplet) for triplet in trigram_tuples]
        
        return trigram_strings  # Returns list of trigram phrases

    def transform(self, texts):
        """Full processing pipeline including trigrams and padding"""
        cleaned_texts = [self.clean_text(text) for text in texts]
        trigram_lists = [self.generate_trigrams(text) for text in cleaned_texts]
        trigram_texts = [' '.join(trigrams) for trigrams in trigram_lists]

        # Convert trigrams to sequences
        sequences = self.tokenizer.texts_to_sequences(trigram_texts)
        
        # Safety check: filter out any token indices that exceed vocabulary size
        if hasattr(self, 'vocab_size'):
            for i in range(len(sequences)):
                sequences[i] = [idx for idx in sequences[i] if idx < self.vocab_size]

        # Pad sequences
        return pad_sequences(sequences, maxlen=self.max_len, padding='post')

print("✅ TrigramTextPreprocessor class defined successfully!")

Stop words defined: 181 words (excluding 28 negations)
✅ TrigramTextPreprocessor class defined successfully!


## analyze_subreddit_sentiment

In [25]:
class RedditSentimentAnalyzer:
    def __init__(self, model_path='best_bilstm_model_final.keras', use_praw=True):
        print("Initializing Reddit Sentiment Analyzer...")
        self.artifact_dir = resolve_artifact_path("")

        tokenizer_path = resolve_artifact_path('fitted_tokenizer.pkl')
        pipeline_path = resolve_artifact_path('text_preprocessing_pipeline.pkl')
        model_path = resolve_artifact_path(model_path)

        # Load the tokenizer
        try:
            with open(tokenizer_path, 'rb') as f:
                self.tokenizer = pickle.load(f)
                if not hasattr(self.tokenizer, 'oov_token'):
                    self.tokenizer.oov_token = "<OOV>"
                print("✅ Tokenizer loaded successfully")
        except Exception as e:
            print(f"❌ Error loading tokenizer: {e}")
            raise

        # Load the preprocessing pipeline
        try:
            with open(pipeline_path, 'rb') as f:
                self.pipeline = pickle.load(f)
            
            self.preprocessor = self.pipeline.named_steps.get('preprocessor')
            if self.preprocessor is None:
                raise ValueError("Pipeline does not contain a 'preprocessor' step")

            # Add vocab_size if missing
            if not hasattr(self.preprocessor, 'vocab_size'):
                self.preprocessor.vocab_size = len(self.tokenizer.word_index) + 1
            
            print("✅ Preprocessing pipeline loaded successfully")
        except Exception as e:
            print(f"❌ Error loading preprocessing pipeline: {e}")
            raise

        # Load the sentiment model
        try:
            self.model = tf.keras.models.load_model(model_path)
            
            # Get embedding layer input dimension for safety checks
            model_config = self.model.get_config()
            self.embedding_dim = None
            for layer in model_config['layers']:
                if layer['class_name'] == 'Embedding':
                    self.embedding_dim = layer['config']['input_dim']
                    break
            
            print("✅ Sentiment model loaded successfully")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise
        
        # Initialize Reddit API if requested
        self.reddit = None
        if use_praw:
            try:
                import praw
                from prawcore.exceptions import ResponseException
                
                self.reddit = praw.Reddit(
                    client_id="xbGJrntKu94WjXFgPx4MgQ",
                    client_secret="7FcKXI-u1TXuWk5gbzDoru-qqlDAhg",
                    user_agent="script:RedditSentimentBot:v1.0 (by u/Internal_Tart_3793)"
                )
                
                # Test connection
                print("Testing Reddit API connection...")
                try:
                    # Try to fetch a couple of posts to verify the connection
                    for i, post in enumerate(self.reddit.subreddit("worldnews").hot(limit=2)):
                        print(f"Title: {post.title[:50]}... | Upvotes: {post.score}")
                    print("\n✅ Reddit API connection successful!")
                except ResponseException as e:
                    print(f"❌ Reddit API error: {e}")
                    self.reddit = None
            except ImportError:
                print("❌ PRAW not installed. Reddit API functionality disabled.")
                self.reddit = None
            except Exception as e:
                print(f"❌ Error initializing Reddit API: {e}")
                self.reddit = None
        
        print("Initialization complete!")

    def _transform_texts(self, texts):
        try:
            return self.pipeline.transform(texts)
        except Exception as pipeline_error:
            print(f"⚠️ Pipeline transform failed, using preprocessor fallback: {pipeline_error}")
            return self.preprocessor.transform(texts)

    def predict_sentiment(self, texts):
        """Predict sentiment for a list of texts"""
        if not texts:
            return []
        
        try:
            # Process texts through the pipeline
            processed = self._transform_texts(texts)
            
            # Double check indices are within embedding dimension bounds
            if self.embedding_dim:
                for i in range(processed.shape[0]):
                    for j in range(processed.shape[1]):
                        if processed[i, j] >= self.embedding_dim:
                            processed[i, j] = 0  # Replace with padding index
            
            # Make predictions
            predictions = self.model.predict(processed)
            
            # Format results
            results = []
            for text, pred in zip(texts, predictions):
                score = float(pred[0])
                sentiment = "Positive" if score > 0.5 else "Negative"
                results.append({
                    'text': text,
                    'score': score,
                    'sentiment': sentiment
                })
            
            return results
        except Exception as e:
            print(f"Error during prediction: {e}")
            return []

    def analyze_subreddit_sentiment(self, subreddit_name, limit=20, sort_by='hot'):
        """Analyze sentiment of posts in a subreddit"""
        if not self.reddit:
            print("❌ Reddit API not available")
            return []
        
        print(f"Analyzing r/{subreddit_name} sentiment (sorting by: {sort_by})...")
        
        try:
            subreddit = self.reddit.subreddit(subreddit_name)
            
            # Get posts based on sort method
            if sort_by.lower() == 'hot':
                posts = subreddit.hot(limit=limit)
            elif sort_by.lower() == 'new':
                posts = subreddit.new(limit=limit)
            elif sort_by.lower() == 'top':
                posts = subreddit.top(limit=limit)
            else:
                print(f"❌ Invalid sort method: {sort_by}. Using 'hot' instead.")
                posts = subreddit.hot(limit=limit)
            
            # Extract post data
            post_data = []
            texts = []
            
            for post in posts:
                # Combine title and selftext for better context
                content = f"{post.title} {post.selftext}"
                texts.append(content)
                
                post_data.append({
                    'id': post.id,
                    'title': post.title,
                    'content': content,
                    'score': post.score,
                    'url': f"https://reddit.com{post.permalink}"
                })
            
            if not texts:
                print(f"⚠️ No posts found in r/{subreddit_name}")
                return []
            
            # Predict sentiment
            print(f"Predicting sentiment for {len(texts)} posts...")
            sentiment_results = self.predict_sentiment(texts)
            if len(sentiment_results) != len(post_data):
                print("⚠️ No usable sentiment predictions were returned")
                return []
            
            # Combine post data with sentiment results
            for i, result in enumerate(sentiment_results):
                post_data[i]['sentiment_score'] = result['score']
                post_data[i]['sentiment'] = result['sentiment']
            
            return post_data
            
        except Exception as e:
            print(f"❌ Error analyzing subreddit: {e}")
            return []

    def analyze_post_comments(self, post_url, limit=50):
        """Analyze sentiment of comments in a specific post"""
        if not self.reddit:
            print("❌ Reddit API not available")
            return []
        
        try:
            # Extract submission ID from URL
            submission_id = post_url.split('comments/')[1].split('/')[0]
            submission = self.reddit.submission(id=submission_id)
            
            print(f"Analyzing comments for post: {submission.title[:50]}...")
            
            # Replace "More Comments" placeholders
            submission.comments.replace_more(limit=None)
            
            # Get comments
            comments = list(submission.comments.list())[:limit]
            
            if not comments:
                print("⚠️ No comments found")
                return []
            
            # Extract comment data
            comment_data = []
            texts = []
            
            for comment in comments:
                if not hasattr(comment, 'body'):  # Skip MoreComments objects
                    continue
                
                texts.append(comment.body)
                
                comment_data.append({
                    'id': comment.id,
                    'content': comment.body,
                    'score': comment.score,
                    'author': str(comment.author),
                    'created_utc': comment.created_utc
                })
            
            # Predict sentiment
            print(f"Predicting sentiment for {len(texts)} comments...")
            sentiment_results = self.predict_sentiment(texts)
            if len(sentiment_results) != len(comment_data):
                print("⚠️ No usable sentiment predictions were returned for comments")
                return []
            
            # Combine comment data with sentiment results
            for i, result in enumerate(sentiment_results):
                comment_data[i]['sentiment_score'] = result['score']
                comment_data[i]['sentiment'] = result['sentiment']
            
            return comment_data
            
        except Exception as e:
            print(f"❌ Error analyzing post comments: {e}")
            return []

    def compare_subreddits(self, subreddit_names, limit_per_sub=20):
        """Compare sentiment across multiple subreddits"""
        if not self.reddit:
            print("❌ Reddit API not available")
            return {}
        
        results = {}
        
        for subreddit in subreddit_names:
            print(f"Analyzing r/{subreddit}...")
            posts = self.analyze_subreddit_sentiment(subreddit, limit=limit_per_sub)
            
            if posts:
                # Calculate aggregate statistics
                sentiment_scores = [post['sentiment_score'] for post in posts]
                positive_count = sum(1 for score in sentiment_scores if score > 0.5)
                
                results[subreddit] = {
                    'posts': posts,
                    'avg_sentiment': sum(sentiment_scores) / len(sentiment_scores),
                    'positive_percentage': (positive_count / len(posts)) * 100,
                    'post_count': len(posts)
                }
        
        return results

    def export_results_to_csv(self, data, filename):
        """Export analysis results to CSV."""
        try:
            df = pd.DataFrame(data)
            df.to_csv(filename, index=False)
            print(f"Results exported to {filename}")
            return True
        except Exception as e:
            print(f"Error exporting results: {e}")
            return False


In [26]:
def analyze_custom_text():
    """Analyze sentiment of custom text input"""
    print("\n--- Custom Text Sentiment Analysis ---")
    
    texts = []
    while True:
        text = input("\nEnter text to analyze (or 'q' to quit): ")
        if text.lower() == 'q':
            break
        texts.append(text)
    
    if not texts:
        print("No texts provided")
        return
    
    print(f"\nAnalyzing {len(texts)} text entries...")
    try:
        pipeline_path = resolve_artifact_path('text_preprocessing_pipeline.pkl')
        model_path = resolve_artifact_path('best_bilstm_model_final.keras')

        with open(pipeline_path, 'rb') as f:
            pipeline = pickle.load(f)
        
        # Add vocab_size if missing
        tokenizer = pipeline.named_steps['preprocessor'].tokenizer
        if not hasattr(pipeline.named_steps['preprocessor'], 'vocab_size'):
            pipeline.named_steps['preprocessor'].vocab_size = len(tokenizer.word_index) + 1
        
        model = tf.keras.models.load_model(model_path)
        
        # Process texts through the pipeline
        try:
            processed = pipeline.transform(texts)
        except Exception as pipeline_error:
            print(f"⚠️ Pipeline transform failed, using preprocessor fallback: {pipeline_error}")
            processed = pipeline.named_steps['preprocessor'].transform(texts)
        
        # Get embedding dimension
        model_config = model.get_config()
        embedding_dim = None
        for layer in model_config['layers']:
            if layer['class_name'] == 'Embedding':
                embedding_dim = layer['config']['input_dim']
                break
        
        # Safety check
        if embedding_dim:
            for i in range(processed.shape[0]):
                for j in range(processed.shape[1]):
                    if processed[i, j] >= embedding_dim:
                        processed[i, j] = 0
        
        # Make predictions
        predictions = model.predict(processed)
        
        # Display results
        print("\nResults:")
        for i, (text, pred) in enumerate(zip(texts, predictions)):
            score = float(pred[0])
            sentiment = "Positive" if score > 0.5 else "Negative" if score < 0.5 else "Neutral"
            confidence = max(score, 1 - score) * 100
            
            print(f"\n--- Text {i+1} ---")
            print(f"Content: {text}")
            print(f"Sentiment: {sentiment}")
            print(f"Score: {score:.4f}")
            print(f"Confidence: {confidence:.2f}%")
    
    except Exception as e:
        print(f"Error analyzing custom text: {e}")


## Main

In [30]:
def main():
    try:
        # Initialize analyzer
        analyzer = RedditSentimentAnalyzer()
        
        while True:
            print("\n--- Reddit Sentiment Analysis Tool ---")
            print("1. Analyze a subreddit")
            print("2. Analyze comments in a specific post")
            print("3. Compare multiple subreddits")
            print("4. Custom text sentiment analysis")
            print("5. Exit")
            
            choice = input("\nEnter your choice (1-5): ")
            
            if choice == '1':
                subreddit = input("Enter subreddit name (without r/): ")
                try:
                    limit = int(input("Enter number of posts to analyze (10-100): "))
                    limit = max(10, min(limit, 100))  # Clamp between 10 and 100
                except ValueError:
                    print("Invalid input, using default of 20 posts")
                    limit = 20
                
                sort_by = input("Sort by (hot/new/top): ").lower()
                if sort_by not in ['hot', 'new', 'top']:
                    sort_by = 'hot'
                
                results = analyzer.analyze_subreddit_sentiment(subreddit, limit, sort_by)
                
                if results:
                    # Calculate aggregate statistics
                    sentiment_scores = [post['sentiment_score'] for post in results]
                    positive_count = sum(1 for score in sentiment_scores if score > 0.5)
                    
                    print(f"\n=== Results for r/{subreddit} ===")
                    print(f"Posts analyzed: {len(results)}")
                    print(f"Average sentiment score: {sum(sentiment_scores) / len(sentiment_scores):.4f}")
                    print(f"Positive posts: {positive_count} ({(positive_count / len(results)) * 100:.1f}%)")
                    print(f"Negative posts: {len(results) - positive_count} ({((len(results) - positive_count) / len(results)) * 100:.1f}%)")
                    
                    # Display top 5 most positive and negative posts
                    sorted_posts = sorted(results, key=lambda x: x['sentiment_score'], reverse=True)
                    
                    print("\n--- Most Positive Posts ---")
                    for i, post in enumerate(sorted_posts[:5]):
                        print(f"{i+1}. {post['title'][:50]}... | Score: {post['sentiment_score']:.4f}")
                    
                    print("\n--- Most Negative Posts ---")
                    for i, post in enumerate(sorted_posts[-5:]):
                        print(f"{i+1}. {post['title'][:50]}... | Score: {post['sentiment_score']:.4f}")

                    export = input("\nExport results to CSV? (y/n): ").lower()
                    if export == 'y':
                        filename = f"sentiment_{subreddit}_{sort_by}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
                        analyzer.export_results_to_csv(results, filename)
            
            elif choice == '2':
                subreddit = input("Enter subreddit name (without r/): ")
                try:
                    post_limit = int(input("Enter number of posts to fetch (1-10): "))
                    post_limit = max(1, min(post_limit, 10))  # Clamp between 1 and 10
                except ValueError:
                    print("Invalid input, using default of 3 posts")
                    post_limit = 3
                
                sort_by = input("Sort posts by (hot/new/top): ").lower()
                if sort_by not in ['hot', 'new', 'top']:
                    sort_by = 'hot'
                
                try:
                    comment_limit = int(input("Enter maximum number of comments to analyze per post (5-50): "))
                    comment_limit = max(5, min(comment_limit, 50))  # Clamp between 5 and 50
                except ValueError:
                    print("Invalid input, using default of 20 comments per post")
                    comment_limit = 20
                
                # First, fetch posts from the subreddit
                print(f"\nFetching {post_limit} {sort_by} posts from r/{subreddit}...")
                
                # Get posts based on sort method
                if not analyzer.reddit:
                    print("❌ Reddit API not available")
                    continue
                
                try:
                    subreddit_obj = analyzer.reddit.subreddit(subreddit)
                    
                    if sort_by.lower() == 'hot':
                        posts = subreddit_obj.hot(limit=post_limit)
                    elif sort_by.lower() == 'new':
                        posts = subreddit_obj.new(limit=post_limit)
                    elif sort_by.lower() == 'top':
                        posts = subreddit_obj.top(limit=post_limit)
                    else:
                        posts = subreddit_obj.hot(limit=post_limit)
                    
                    # Convert to list to get fixed number of posts
                    posts = list(posts)
                    
                    if not posts:
                        print(f"No posts found in r/{subreddit}")
                        continue
                    
                    # Display fetched posts and let user select
                    print(f"\nFetched {len(posts)} posts from r/{subreddit}:")
                    all_comment_results = []
                    
                    for idx, post in enumerate(posts):
                        print(f"\n{idx+1}. {post.title}")
                        print(f"   URL: https://reddit.com{post.permalink}")
                        print(f"   Upvotes: {post.score} | Comments: {post.num_comments}")
                        
                        # Process comments for this post
                        print(f"\n   Analyzing up to {comment_limit} comments...")
                        
                        # Get the submission to access comments
                        submission = analyzer.reddit.submission(id=post.id)
                        
                        # Replace "More Comments" placeholders
                        submission.comments.replace_more(limit=2)  # Limit to avoid excessive API calls
                        
                        # Get comments
                        comments = list(submission.comments.list())[:comment_limit]
                        
                        if not comments:
                            print("   ⚠️ No comments found")
                            continue
                        
                        # Extract comment data
                        comment_data = []
                        texts = []
                        
                        for comment in comments:
                            if not hasattr(comment, 'body'):  # Skip MoreComments objects
                                continue
                            
                            texts.append(comment.body)
                            
                            comment_data.append({
                                'id': comment.id,
                                'post_title': post.title,
                                'post_url': f"https://reddit.com{post.permalink}",
                                'content': comment.body,
                                'score': comment.score,
                                'author': str(comment.author),
                                'created_utc': comment.created_utc
                            })
                        
                        if not texts:
                            print("   ⚠️ No valid comments found")
                            continue
                        
                        # Predict sentiment
                        sentiment_results = analyzer.predict_sentiment(texts)
                        
                        # Combine comment data with sentiment results
                        for i, result in enumerate(sentiment_results):
                            comment_data[i]['sentiment_score'] = result['score']
                            comment_data[i]['sentiment'] = result['sentiment']
                        
                        # Calculate aggregate statistics for this post's comments
                        sentiment_scores = [comment['sentiment_score'] for comment in comment_data]
                        positive_count = sum(1 for score in sentiment_scores if score > 0.5)
                        
                        print(f"   Comments analyzed: {len(comment_data)}")
                        print(f"   Average sentiment: {sum(sentiment_scores) / len(sentiment_scores):.4f} " +
                              f"(Positive: {positive_count}, Negative: {len(comment_data) - positive_count})")
                        
                        # Add to overall results
                        all_comment_results.extend(comment_data)
                    
                    if all_comment_results:
                        # Calculate aggregate statistics for all comments
                        all_sentiment_scores = [comment['sentiment_score'] for comment in all_comment_results]
                        all_positive_count = sum(1 for score in all_sentiment_scores if score > 0.5)
                        
                        print(f"\n=== Overall Comment Analysis Results for r/{subreddit} ===")
                        print(f"Total comments analyzed: {len(all_comment_results)}")
                        print(f"Average sentiment score: {sum(all_sentiment_scores) / len(all_comment_results):.4f}")
                        print(f"Positive comments: {all_positive_count} ({(all_positive_count / len(all_comment_results)) * 100:.1f}%)")
                        print(f"Negative comments: {len(all_comment_results) - all_positive_count} " +
                              f"({((len(all_comment_results) - all_positive_count) / len(all_comment_results)) * 100:.1f}%)")
                        
                        # Display top 3 most positive and negative comments
                        sorted_comments = sorted(all_comment_results, key=lambda x: x['sentiment_score'], reverse=True)
                        
                        print("\n--- Most Positive Comments ---")
                        for i, comment in enumerate(sorted_comments[:3]):
                            print(f"{i+1}. \"{comment['content'][:100]}...\"")
                            print(f"   Post: {comment['post_title'][:50]}...")
                            print(f"   Score: {comment['sentiment_score']:.4f}")
                        
                        print("\n--- Most Negative Comments ---")
                        for i, comment in enumerate(sorted_comments[-3:]):
                            print(f"{i+1}. \"{comment['content'][:100]}...\"")
                            print(f"   Post: {comment['post_title'][:50]}...")
                            print(f"   Score: {comment['sentiment_score']:.4f}")
                        
                        export = input("\nExport all comment results to CSV? (y/n): ").lower()
                        if export == 'y':
                            filename = f"comments_{subreddit}_{sort_by}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
                            analyzer.export_results_to_csv(all_comment_results, filename)
                
                except Exception as e:
                    print(f"❌ Error analyzing subreddit posts and comments: {e}")
        
            
            elif choice == '3':
                subreddits_input = input("Enter subreddit names separated by commas (e.g. AskReddit,worldnews,gaming): ")
                subreddits = [sub.strip() for sub in subreddits_input.split(',')]
                
                if not subreddits:
                    print("No valid subreddits provided")
                    continue
                
                try:
                    limit = int(input("Enter number of posts per subreddit (10-50): "))
                    limit = max(10, min(limit, 50))  # Clamp between 10 and 50
                except ValueError:
                    print("Invalid input, using default of 20 posts per subreddit")
                    limit = 20
                
                results = analyzer.compare_subreddits(subreddits, limit)
                
                if results:
                    print("\n=== Subreddit Comparison Results ===")
                    print(f"{'Subreddit':<15} | {'Avg Sentiment':<15} | {'Positive %':<10} | {'Posts':<5}")
                    print("-" * 50)
                    
                    for sub, data in results.items():
                        print(f"{sub:<15} | {data['avg_sentiment']:<15.4f} | {data['positive_percentage']:<10.1f}% | {data['post_count']:<5}")
            
            elif choice == '4':
                analyze_custom_text()
            
            elif choice == '5':
                print("Exiting the program. Goodbye!")
                break
            
            else:
                print("Invalid choice. Please enter a number between 1 and 5.")
    
    except Exception as e:
        print(f"An error occurred in the main function: {e}")

if __name__ == "__main__":
    main()

Initializing Reddit Sentiment Analyzer...
✅ Tokenizer loaded successfully
✅ Preprocessing pipeline loaded successfully
✅ Sentiment model loaded successfully
Testing Reddit API connection...
Title: /r/WorldNews Live Thread: Russian Invasion of Ukra... | Upvotes: 404
Title: /r/WorldNews Discussion Thread: US and Israel laun... | Upvotes: 251

✅ Reddit API connection successful!
Initialization complete!

--- Reddit Sentiment Analysis Tool ---
1. Analyze a subreddit
2. Analyze comments in a specific post
3. Compare multiple subreddits
4. Custom text sentiment analysis
5. Exit

--- Custom Text Sentiment Analysis ---

Analyzing 4 text entries...
⚠️ Pipeline transform failed, using preprocessor fallback: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 509ms/step

Results:

--- Text 1 ---
Content: very good food, totally reccomend going there
Sentiment: Negative
Score: 0.3516
Confidence: 64.84%

--- Text 2

In [28]:
print("\n--- Validation ---")
validator = RedditSentimentAnalyzer(use_praw=False)
sample_texts = ["This is a great post but not perfect"]
print("Transform shape:", validator._transform_texts(sample_texts).shape)
print("Prediction sample:", validator.predict_sentiment(sample_texts))


--- Validation ---
Initializing Reddit Sentiment Analyzer...
✅ Tokenizer loaded successfully


/Users/mohammedwalidadawy/Development/Social-Media-Sentiment-Analysis/.venv/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Preprocessing pipeline loaded successfully
✅ Sentiment model loaded successfully
Initialization complete!
⚠️ Pipeline transform failed, using preprocessor fallback: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
Transform shape: (1, 100)
⚠️ Pipeline transform failed, using preprocessor fallback: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 793ms/step
Prediction sample: [{'text': 'This is a great post but not perfect', 'score': 0.6171262860298157, 'sentiment': 'Positive'}]
